# PydanticAI together with Pydantic Models

In [1]:
from model import nividia_model
from pydantic_ai import Agent

employee_simulator = Agent(
    model=nividia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions="Du är expert på att simulera anställda i Sverige. Simulerata i detta formatet likt exemplet"
    """
        name:"Berra",
        age 40
        salary: 35000,
        role: "IT-supporter"
        ort: "Göteborg"
    """
    )

result = await employee_simulator.run("Ge mig en data engineer som jobbar i stockholm")

print(result.output)

name: "Sofia Lindqvist",
age: 29,
salary: 54000,
role: "Data Engineer",
ort: "Stockholm"


## Force a structured output using pydantic

In [3]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str = Field(description="för och efternamn, inte jättelångt, svenska eller internationella namn")
    age: int = Field(description="ålder från 18 till 67 år", ge=18, le=67)
    salary: int = Field(description="lön mellan 30k-60k, ju mer senior position desto högre lön", 
                        ge=30000, 
                        le=60000
                    )
    role: str = Field(description="yrke/roll")
    description: str = Field(description="Kort beskrivning inom yrket")

employee1 = EmployeeModel(name="Bertil", age=40, salary=30000, role="IT support", description="Hjälper andra anställda med IT")

employee1

EmployeeModel(name='Bertil', age=40, salary=30000, role='IT support', description='Hjälper andra anställda med IT')

In [5]:
employee_simulator = Agent(
    model=nividia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions="Du är expert på att simulera anställda i Sverige"
    )

result = await employee_simulator.run("Ge mig en data engineer i stockholm")

result.output

'Hej! Jag är **Erik Johansson**, 32 år, och jobbar som **Senior Data Engineer** i Stockholm. Här är en simulerad presentation av hur en sådan roll kan se ut i svensk kontext.\n\n---\n\n### 👤 Grundfakta\n- **Namn:** Erik Johansson  \n- **Ålder:** 32  \n- **Bor:** Södermalm, Stockholm (en gammal 2:a som han fick genom bostadskön efter 4 år)  \n- **Utbildning:** Civilingenjör i datateknik, KTH  \n- **Roll:** Senior Data Engineer på en fintech-scaleup i Hammarby Sjöstad  \n- **Anställning:** Tillsvidare, kollektivavtal (Unionen/Almega IT-tjänstemän)  \n- **Lön:** Cirka 68 000 SEK/månad före skatt  \n- **Språk:** Svenska och engelska (arbetsplatsen är internationell, möten på engelska)  \n- **Pendlar:** Tunnelbanans röda linje till Hammarbyhöjden, eller cyklar när det inte är halt\n\n---\n\n### 🛠 Tech stack (typisk för Stockholm)\n- **Språk:** Python, SQL, Scala (lite äldre legacy), Java  \n- **Data:** Apache Spark, dbt, Snowflake, BigQuery, Kafka  \n- **Orchestration:** Apache Airflow, Pre

In [6]:
result = await employee_simulator.run("Ge mig en data engineer i stockholm", output_type=EmployeeModel)

In [11]:
employee2 = result.output

In [12]:
type(employee2)

__main__.EmployeeModel

In [14]:
employee2.name + " är cool"

'Erik Andersson är cool'

In [15]:
employee2.salary

52000

In [16]:
for key, value in employee2.__dict__.items():
    print("{key} : {value}")

{key} : {value}
{key} : {value}
{key} : {value}
{key} : {value}
{key} : {value}


## Simulate many employees

In [18]:
result = await employee_simulator.run("Ge mig totalt 10 anställada inom IT/HR/marketing", output_type=list[EmployeeModel])

result.output

[EmployeeModel(name='Erik Johansson', age=38, salary=52000, role='Systemutvecklare', description='Utvecklar webbapplikationer i .NET och Azure med fokus på kodkvalitet och testdriven utveckling.'),
 EmployeeModel(name='Sofia Nilsson', age=29, salary=36000, role='IT-supportspecialist', description='Ansvarar för första linjens support, felsökning av hårdvara och mjukvara samt användarutbildning.'),
 EmployeeModel(name='Amir Hassan', age=45, salary=58000, role='Molnarkitekt', description='Designar och förvaltar skalbara molnlösningar på AWS och Azure med säkerhet och kostnadskontroll i fokus.'),
 EmployeeModel(name='Lisa Andersson', age=34, salary=41000, role='HR-generalist', description='Hanterar rekrytering, arbetsrättsliga frågor, lönerevisioner och medarbetarsamtal i ett medelstort företag.'),
 EmployeeModel(name='Maria Karlsson', age=31, salary=39000, role='Rekryteringskonsult', description='Driver rekryteringsprocesser för IT- och tjänstesektorn, från annonsering till anställningsav

In [19]:
employees = result.output

employees

[EmployeeModel(name='Erik Johansson', age=38, salary=52000, role='Systemutvecklare', description='Utvecklar webbapplikationer i .NET och Azure med fokus på kodkvalitet och testdriven utveckling.'),
 EmployeeModel(name='Sofia Nilsson', age=29, salary=36000, role='IT-supportspecialist', description='Ansvarar för första linjens support, felsökning av hårdvara och mjukvara samt användarutbildning.'),
 EmployeeModel(name='Amir Hassan', age=45, salary=58000, role='Molnarkitekt', description='Designar och förvaltar skalbara molnlösningar på AWS och Azure med säkerhet och kostnadskontroll i fokus.'),
 EmployeeModel(name='Lisa Andersson', age=34, salary=41000, role='HR-generalist', description='Hanterar rekrytering, arbetsrättsliga frågor, lönerevisioner och medarbetarsamtal i ett medelstort företag.'),
 EmployeeModel(name='Maria Karlsson', age=31, salary=39000, role='Rekryteringskonsult', description='Driver rekryteringsprocesser för IT- och tjänstesektorn, från annonsering till anställningsav

In [ ]:
salary = 0
for employee in employees:
    salary += employee.salary

salary/len(employees)

46100.0

## Convert to dataframe


In [24]:
employees[0].model_dump()

{'name': 'Erik Johansson',
 'age': 38,
 'salary': 52000,
 'role': 'Systemutvecklare',
 'description': 'Utvecklar webbapplikationer i .NET och Azure med fokus på kodkvalitet och testdriven utveckling.'}

In [25]:
import pandas as pd

df = pd.DataFrame([employee.model_dump() for employee in employees])
df.head()

,name,age,salary,role,description
0,Erik Johansson,38,52000,Systemutvecklare,Utvecklar webbapplikationer i .NET och Azure m...
1,Sofia Nilsson,29,36000,IT-supportspecialist,"Ansvarar för första linjens support, felsöknin..."
2,Amir Hassan,45,58000,Molnarkitekt,Designar och förvaltar skalbara molnlösningar ...
3,Lisa Andersson,34,41000,HR-generalist,"Hanterar rekrytering, arbetsrättsliga frågor, ..."
4,Maria Karlsson,31,39000,Rekryteringskonsult,Driver rekryteringsprocesser för IT- och tjäns...


In [26]:
df.to_csv("employees.csv", index=None)